In [ ]:
import panel as pn
import json as js

import requests as rq

: 

In [ ]:
import reqESPN

: 

# Get Team Data

In [ ]:
tm = reqESPN.getStat_frTm(2023, 1)

: 

In [ ]:
tm = js.loads(tm)

: 

In [ ]:
pn.extension()

pn.pane.JSON(tm)

: 

In [ ]:
athRef = tm['athletes']['$ref']

: 

In [ ]:
r = rq.get(athRef)

ath = js.loads(r.text)

: 

The ath variable contains a list of API endpoints w/ `playerid` for all


In [ ]:
ath


: 

In [ ]:
# Link names in tm variable

lnk_nms = [x['text'] for x in tm['links']]

lnk_nms

: 

In [ ]:
hrefs = [x['href'] for x in tm['links']]

# make get request for the injuries link
r = rq.get(hrefs[-2])

: 

### the request fails
see output of below cell

In [ ]:
from pprint import pprint
pprint(r.content)

: 

In [ ]:
lamar_id = 3916387
plyr = js.loads(reqESPN.getPlyr_frID(lamar_id))

: 

## Output of pprint( plyr )

In [ ]:
from pprint import pprint

pprint(plyr)

: 

# Headshot

* Headshot object, 
    * hsob
* reference to headshot, 
    * hsref
* headshot response, 
    * hsres

In [ ]:

hsob = plyr['athlete']['headshot']
hsref = hsob['href']
hsres = rq.get(hsref)


: 

In [ ]:
hsref

: 

# A digression into PIL library (Python Image Lib)

In [ ]:
from PIL import Image
from io import BytesIO

i = Image.open(BytesIO(hsres.content)).convert('RGB')

: 

In [ ]:
i

: 

# TODO :: Use an image processing solution to remove blur behind lamar

# Panel will create panes from URLs!

In [ ]:
pn.extension("Bokeh")
# open the PIL Image in panel

hs = pn.pane.PNG(hsref)

: 

In [ ]:
from panel.layout.gridstack import GridStack

gs = GridStack(sizing_mode='stretch_both', min_height=600)

gs[0:2, 0:1] = hs
gs[0:2, 1:2] = pn.pane.Markdown("## Hello World")
gs.visible = False

: 

In [ ]:
lamar_stat = reqESPN.getStats_frID(lamar_id)

: 

In [ ]:
stat_js = js.loads(lamar_stat)

: 

In [ ]:
stat_js.keys()

: 

In [ ]:
filts = stat_js['filters']

teams = stat_js['teams']

cats = stat_js['categories']

gloss = stat_js['glossary']

: 

# Categories
* `labels` for stat names
* `statistics` for stats by year
* `totals` for career totals

In [ ]:
# Labels for Statistical Category
# Labels are not the same for each category, so this will need to 
# be decided within the category cleaning function
cat_labels = cats[0]['labels']

: 

In [ ]:
f = lambda x : zip(cat_labels, x)
cat_stat = f(cats[0]['statistics'])
cat_tot = f(cats[0]['totals'])

: 

: 

In [ ]:
# sample stat object from category
## Stats for the passing category
tst = cats[0]['statistics'][0]['stats']

## zip to dict generator for a stat list from a category
zipStats = lambda statlst, lbl : { k : v for k, v in zip(lbl, statlst)} 
zipStats(tst, cat_labels)

: 

In [ ]:
# A list of stat dicts is contained for each category
pprint(list(cats[0].keys()))

: 

In [ ]:
cats[1].keys()

: 

: 

In [ ]:
careerRush = zipStats( cats[1]['totals'] , cats[1]['labels'])
statPack = { x['name'] : zipStats(x['totals'], x['labels']) for x in cats }

: 

In [ ]:
careerTotals = zipStats( cats[0]['totals'] , cats[0]['labels'])
# gs[3:5, 0:2] = pn.pane.JSON(careerTotals)
gs[3:5, 2:4] = pn.pane.JSON(statPack)

: 

In [ ]:
gs.visible = True
gs

: 

In [ ]:
pprint(list(statPack.keys()))


: 

In [ ]:
statPack['passing'].keys()

: 

In [ ]:
ky = {'passing' : 'ps_',
 'rushing' : 'rs_',
 'receiving' : 'rc_',
 'defensive' : 'df_',
 'scoring' : 'pt_',
 'kicking' : 'k_'}


: 

In [ ]:
cats[0]['statistics']

: 

In [ ]:
dispZIPtotals = { k : v for k, v in zip(cats[0]['displayNames'], cats[0]['totals']) } 



: 

In [ ]:
def genCategoricalStatDicts(cat):
    return { x['name'] : zipStats(x['totals'], x['labels']) for x in cat }

: 

In [ ]:
genCategoricalStatDicts(cats)

: 

In [ ]:
dispZIPtotals

: 

In [ ]:
ky

: 

In [ ]:
dispZIPtotals

: 

In [ ]:
categoricalStats = list( zip(ky.items(), cats) )

: 

In [ ]:
# Category Name (disp)
dispname = categoricalStats[0][0][0]
# for k in categoricalStats[0] : dname = k[0][0]

# Category Key (ky)
catkey = categoricalStats[0][0][1]
# for k in categoricalStats[0] : ckey = k[0][1]

# Category Stats (cstats)
cstat = categoricalStats[0][1]
# for k in categoricalStats[0] : cstat = k[1]

## APPLY CLEANING METHOD FROM ABOVE
# TODO:: WRITE CLEANING METHOD


: 

In [ ]:
cstat

: 